In [1]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import warnings

warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('elviscota2023obweekday_export_odbc.csv')

In [3]:
ke_df=pd.read_excel('COTA_KINGElvis.xlsx',sheet_name='Elvis_Review')

In [4]:
ke_df.head()

,Elvis_Date,elvis_id,1st Cleaner,FINAL_REVIEWER,Final_Usage,REASON FOR REMOVAL,REASON FOR REMOVAL [Other],route_match_flag,distance_flag,SUPERVISOR_COMMENT,...,2X_REVIEW_CHECK.1,2x_REVIEWED_BY,2x_REVIEWED_FLAG,ADMIN_APPROVED,SURVEY_RECOVERY,SURVEY_RECOVERY_REVIEWED_BY,Recovery Check,2x_REVIEWED_BY.1,2x_REVIEWED_FLAG.1,ADMIN_APPROVED.1
0,2023-11-03,5567,Test/No 5 MIN,NaN,Remove,Test/No 5 MIN,,Elvis_Review,Elvis_Review,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2023-11-03,5568,Test/No 5 MIN,NaN,Remove,Test/No 5 MIN,,Elvis_Review,Elvis_Review,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2023-11-03,5570,Test/No 5 MIN,NaN,Remove,Test/No 5 MIN,,Elvis_Review,Elvis_Review,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2023-11-03,5573,Test/No 5 MIN,NaN,Remove,Test/No 5 MIN,,Elvis_Review,Elvis_Review,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2023-11-03,5575,Test/No 5 MIN,NaN,Remove,Test/No 5 MIN,,Elvis_Review,Elvis_Review,NaN,...,NaT,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [5]:
ke_df[['FINAL_REVIEWER']].value_counts()

FINAL_REVIEWER             
Sneh                           427
Divya sravani                  378
Bhumika                        371
Gowtham                        339
Kesar                          284
Sameera                        272
Saurav                         190
Vaishnavi                      189
Himanshu                       183
Trusha                         182
Sakshi                         182
Raja                           182
T A Divya                      149
Kalpesh                        131
Karthik                        106
muskan                         100
Priyanka                        98
karthik                         88
Srisamhita                      88
Rutwa                           71
Saurav                          50
Divya sravani, Brian            25
Bhumika, Brian                  24
Sakshi, Brian                   24
Karthik,Regina, Brian           21
Gowtham, Brian                  19
Gowtham,Regina, Brian           16
Divya sravani,Regina, Brian

In [6]:
df=pd.merge(df,ke_df[['id','FINAL_REVIEWER','Final_Usage']],on='id',how='left')

In [7]:
df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,ELVIS_USER_CHANGE_7_C_REASON,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,_REVERSE_TRIP,GENERATABLE_TRIPS,FINAL_REVIEWER,Final_Usage
0,5567,2023-10-26 16:30:15,78,en,2023-10-26 16:29:55,2023-10-26 16:30:15,70.113.126.123,https://cota-oh.etc-research.com/,-14400,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remove
1,5568,2023-10-26 17:00:33,78,en,2023-10-26 17:00:12,2023-10-26 17:00:33,70.190.112.58,https://cota-oh.etc-research.com/index.php/adm...,-14400,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remove
2,5570,2023-10-26 17:01:13,78,en,2023-10-26 17:00:57,2023-10-26 17:01:13,70.190.112.58,https://cota-oh.etc-research.com/index.php/adm...,-14400,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remove
3,5573,2023-10-26 17:10:18,78,en,2023-10-26 17:04:26,2023-10-26 17:10:18,70.113.126.123,NaN,-14400,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remove
4,5575,2023-10-26 17:13:04,78,en,2023-10-26 17:10:22,2023-10-26 17:13:04,70.113.126.123,https://cota-oh.etc-research.com/index.php/sur...,-14400,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remove


In [8]:
df.shape

(5581, 747)

In [9]:
def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns


In [10]:
def get_distance_between_coordinates(lat1, lon1, lat2, lon2):
    try:
        lat1 = float(lat1)
        lon1 = float(lon1)
        lat2 = float(lat2)
        lon2 = float(lon2)
        
        coords_1 = (lat1, lon1)
        coords_2 = (lat2, lon2)
        
        distance = geodesic(coords_1, coords_2).miles
        return distance
    except (ValueError, TypeError) as e:
        # Handle the exception here
        print(f"Error calculating distance: {e}")  # Change to the desired distance unit

In [11]:
df['Origin_Destin_Distance']=None
df['Boarding_Alighting_Distance']=None

In [12]:
origin_destin_columns_check=['originaddresslat', 'originaddresslong','destinaddresslat', 'destinaddresslong']
origin_destin_columns=check_all_characters_present(df,origin_destin_columns_check)
origin_destin_columns.sort()

In [13]:
'''
    origin_destin_columns=['DESTIN_ADDRESS_LAT',0
                    'DESTIN_ADDRESS_LONG',1
                    'ORIGIN_ADDRESS_LAT',2
                    'ORIGIN_ADDRESS_LONG'3
                    ]
'''

"\n    origin_destin_columns=['DESTIN_ADDRESS_LAT',0\n                    'DESTIN_ADDRESS_LONG',1\n                    'ORIGIN_ADDRESS_LAT',2\n                    'ORIGIN_ADDRESS_LONG'3\n                    ]\n"

In [14]:

boarding_alighting_columns_check=['stoponlat', 'stoponlong', 'stopofflat', 'stopofflong']
boarding_alighting_columns=check_all_characters_present(df,boarding_alighting_columns_check)
boarding_alighting_columns.sort()


In [15]:

'''
    boarding_alighting_columns=[
        'STOP_OFF_LAT',0
        'STOP_OFF_LONG',1
        'STOP_ON_LAT',2
        'STOP_ON_LONG'3
        ]
'''

"\n    boarding_alighting_columns=[\n        'STOP_OFF_LAT',0\n        'STOP_OFF_LONG',1\n        'STOP_ON_LAT',2\n        'STOP_ON_LONG'3\n        ]\n"

In [16]:
df.dropna(subset=[*origin_destin_columns,*boarding_alighting_columns],how='any',inplace=True)

In [17]:
df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,_REVERSE_TRIP,GENERATABLE_TRIPS,FINAL_REVIEWER,Final_Usage,Origin_Destin_Distance,Boarding_Alighting_Distance
5,5576,2023-10-26 17:23:27,78,en,2023-10-26 17:13:06,2023-10-26 17:23:27,70.113.126.123,https://cota-oh.etc-research.com/index.php/sur...,-14400,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remove,None,None
8,5606,2023-10-30 00:58:37,78,en,2023-10-30 00:53:27,2023-10-30 00:58:37,70.190.112.58,https://cota-oh.etc-research.com/index.php/adm...,-14400,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Remove,None,None
24,5646,2023-10-30 12:44:33,78,en,2023-10-30 12:35:35,2023-10-30 12:44:33,174.207.40.240,https://cota-oh.etc-research.com/,-14400,CDK,...,NaN,NaN,NaN,NaN,NaN,NaN,"T A Divya, Brian, Regina",Use,None,None
28,5657,2023-10-30 13:33:18,78,en,2023-10-30 13:11:37,2023-10-30 13:33:18,174.207.40.240,https://cota-oh.etc-research.com/,-14400,CDK,...,NaN,NaN,NaN,NaN,NaN,NaN,"T A Divya, Regina",Use,None,None
42,5681,2023-10-30 14:14:40,78,en,2023-10-30 14:07:48,2023-10-30 14:14:40,174.193.122.81,https://cota-oh.etc-research.com/index.php/sur...,-14400,BLN,...,NaN,NaN,NaN,NaN,NaN,NaN,"T A Divya, Regina",Use,None,None


In [18]:
df.shape

(1552, 749)

In [19]:
for index, row in df.iterrows():
    df.loc[index,'Origin_Destin_Distance']=get_distance_between_coordinates(row[origin_destin_columns[2]],row[origin_destin_columns[3]],row[origin_destin_columns[0]],row[origin_destin_columns[1]])
    df.loc[index,'Boarding_Alighting_Distance']=get_distance_between_coordinates(row[boarding_alighting_columns[2]],row[boarding_alighting_columns[3]],row[boarding_alighting_columns[0]],row[boarding_alighting_columns[1]])

In [20]:
df[['Origin_Destin_Distance','Boarding_Alighting_Distance']]

,Origin_Destin_Distance,Boarding_Alighting_Distance
5,1922.520388,4.000293
8,16.07788,1.175554
24,0.0,4.48932
28,0.259203,0.22091
42,6.101705,5.737757
...,...,...
5549,2.719725,1.837637
5550,5.050222,4.815363
5567,4.971223,4.93682
5568,1.370425,1.220114


In [21]:
df[[*origin_destin_columns,*boarding_alighting_columns]]

,DESTIN_ADDRESS_LAT,DESTIN_ADDRESS_LONG,ORIGIN_ADDRESS_LAT,ORIGIN_ADDRESS_LONG,STOP_OFF_LAT,STOP_OFF_LONG,STOP_ON_LAT,STOP_ON_LONG
5,34.424000,-117.370173,39.974567,-82.987582,39.982483,-82.856942,40.035014,-82.888855
8,39.982620,-83.149198,39.817381,-82.935846,39.965149,-83.001318,39.981998,-83.004613
24,39.955255,-82.881798,39.955255,-82.881798,39.955554,-82.882520,39.957720,-82.967026
28,39.955547,-82.938143,39.953133,-82.941884,39.957238,-82.937905,39.957373,-82.942062
42,40.050827,-83.029741,39.968780,-82.986813,40.050198,-83.020077,39.968247,-83.001678
...,...,...,...,...,...,...,...,...
5549,39.961094,-83.034554,39.994006,-83.006352,39.968215,-83.007780,39.994834,-83.006583
5550,40.013666,-82.967384,40.080646,-82.928970,40.013397,-82.967664,40.082116,-82.951789
5567,39.960808,-82.999279,40.031837,-83.015020,39.960277,-83.000377,40.030843,-83.015789
5568,39.975582,-83.012655,39.995013,-83.007298,39.978411,-83.003898,39.995886,-83.007426


In [22]:
df[['Origin_Destin_Distance']]

,Origin_Destin_Distance
5,1922.520388
8,16.07788
24,0.0
28,0.259203
42,6.101705
...,...
5549,2.719725
5550,5.050222
5567,4.971223
5568,1.370425


 1. Is there a reviewer making a ton of changes
 2. Is there a reviewer not making any changes and we're getting a lot of our flags on their records to review again
 3. Is there a certain route that is getting a lot of changes
 4. Overall summary (Previous transfers-20% are being changed)

In [23]:
df.shape

(1552, 749)

In [25]:
df=(df[df['INTERV_INIT']!='999'])

In [27]:
df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,_REVERSE_TRIP,GENERATABLE_TRIPS,FINAL_REVIEWER,Final_Usage,Origin_Destin_Distance,Boarding_Alighting_Distance
24,5646,2023-10-30 12:44:33,78,en,2023-10-30 12:35:35,2023-10-30 12:44:33,174.207.40.240,https://cota-oh.etc-research.com/,-14400,CDK,...,NaN,NaN,NaN,NaN,NaN,NaN,"T A Divya, Brian, Regina",Use,0.0,4.48932
28,5657,2023-10-30 13:33:18,78,en,2023-10-30 13:11:37,2023-10-30 13:33:18,174.207.40.240,https://cota-oh.etc-research.com/,-14400,CDK,...,NaN,NaN,NaN,NaN,NaN,NaN,"T A Divya, Regina",Use,0.259203,0.22091
42,5681,2023-10-30 14:14:40,78,en,2023-10-30 14:07:48,2023-10-30 14:14:40,174.193.122.81,https://cota-oh.etc-research.com/index.php/sur...,-14400,BLN,...,NaN,NaN,NaN,NaN,NaN,NaN,"T A Divya, Regina",Use,6.101705,5.737757
47,5686,2023-10-30 14:30:15,78,en,2023-10-30 14:17:39,2023-10-30 14:30:15,174.207.40.240,https://cota-oh.etc-research.com/index.php/sur...,-14400,CDK,...,NaN,NaN,NaN,NaN,NaN,NaN,T A Divya,Use,5.269188,5.167648
49,5689,2023-10-30 14:30:31,78,en,2023-10-30 14:25:32,2023-10-30 14:30:31,107.77.192.128,https://cota-oh.etc-research.com/index.php/sur...,-14400,JTC,...,NaN,NaN,NaN,NaN,NaN,NaN,T A Divya,Use,5.152383,5.165863


In [31]:
df[['FINAL_REVIEWER','Final_Usage']].value_counts()

FINAL_REVIEWER               Final_Usage
Saurav                       Use            170
Divya sravani                Use            132
Gowtham                      Use            114
Sneh                         Use            106
Bhumika                      Use            100
Kesar                        Use             94
Karthik                      Use             81
Sameera                      Use             80
Himanshu                     Use             67
Trusha                       Use             66
Raja                         Use             58
Kalpesh                      Use             56
Vaishnavi                    Use             56
T A Divya                    Use             50
Sakshi                       Use             50
Saurav                       Use             47
karthik                      Use             30
Srisamhita                   Use             23
Priyanka                     Use             14
Sakshi                       use             11

In [41]:
names={}
names_list=['Saurav','Saurav  ','Sameera','Sneh ','T A Divya',' Divya sravani ']

In [42]:
for name in names_list:
    name=name.strip()
    names[name]=names.get(name,0)+1
names

{'Saurav': 2, 'Sameera': 1, 'Sneh': 1, 'T A Divya': 1, 'Divya sravani': 1}

In [43]:
max(names,key=names.get)

'Saurav'

In [54]:
data='Kesar.Regina, Brian'

In [61]:
if '.' in data:
    data_list=data.split('.')
    data=','.join(data_list)


'Kesar,Regina, Brian'

In [59]:
data_list=data.split(',')

In [60]:
data_list

['Kesar', 'Regina', ' Brian']